# CRE — heterogeneous full-system soak test

This is a **shakedown, not a sample, benchmark, prevalence estimate, or example hunt**. Its deliverable is a durable failure census: input-shape failures, transport outcomes, provider failures, parse quarantines, every disposition, per-stratum reachability, crashes, integrity checks, and the incidental findings that happen to appear.

Diagnostic rates describe only this deliberately adversarial corpus. They must never be quoted as population rates or precision figures.

**Pinned engine:** merge/f2-into-f3f7 at 0a8e663b926af94f7da8d67964118775ab580268.

## Run order

1. Restart the Colab runtime.
2. Run Cells 1–5. They make no Anthropic calls.
3. Read the input profile and cost gate.
4. Leave ENABLE_PAID_RUN false until the free stages are satisfactory.
5. Run Cells 6–10 for the current chunk.
6. Change CHUNK_INDEX in Cell 3 and repeat.

The notebook preserves every failed attempt. Retries handle transient provider failures, and a chunk contains only three papers, so an unrecoverable failure has a bounded blast radius. Re-running Cell 3 after an interrupted paid stage automatically starts a new attempt directory without deleting the old one.

The paid full-text coverage stage uses Anthropic's explicit 5-minute prompt cache only for the invariant instruction prefix. The rendered prompt is reconstructed and asserted byte-for-byte before every paid request; the model, evidence, claim, output limit, parser, and verifier gates are unchanged.

## Taxonomy scope

F1 and F2 run in Band 1. F3, F4, and F6 run against retrieved PMC full text. F5 and F7 have no production evidence builders and are reported unreachable. The PubMed retracted-publication lookup is recorded as an F8 precursor, but taxonomy-complete F8 is reported **not implemented** because the required timing detector is absent.

Reachability always prints before findings. A zero is interpreted only against the number of records or claims that actually reached that stratum.

## Cell 1 — pinned bootstrap and durable logging helpers

In [ ]:
# Free stage. Installs pinned dependencies, mounts Drive, and checks out the exact engine.
import os, sys, json, re, time, hashlib, threading, subprocess, importlib
import collections, traceback, resource, shutil, statistics, xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse

BRANCH = "merge/f2-into-f3f7"
EXPECTED_COMMIT = "0a8e663b926af94f7da8d67964118775ab580268"
EXPECTED_BAND_PROMPTS_BLOB = "fa01126e2b9482d450065fd70cd0eb1fea816f5c"
REPO_URL = "https://github.com/astonliu/citation-repair-engine.git"
REPO = Path("/content/cre-mass-hunt")
PKG_ROOT = REPO / "citation_repair_F1_handoff"
EMAIL = "aston.hliu@gmail.com"
MODEL = "claude-opus-5"

subprocess.run([
    sys.executable, "-m", "pip", "-q", "install",
    "rapidfuzz==3.14.5", "requests==2.32.5", "lxml==6.0.2",
    "anthropic==0.122.0", "jsonschema==4.26.0", "pytest==9.1.1",
], check=True)

from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def get_secret(name, required=False):
    try:
        value = userdata.get(name) or ""
    except Exception:
        value = ""
    if required and not value:
        raise RuntimeError(f"{name} is missing from Colab Secrets")
    return value

def read_jsonl(path):
    p = Path(path)
    if not p.exists():
        return []
    return [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines()
            if line.strip()]

def append_jsonl(path, record):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n")
        fh.flush()
        os.fsync(fh.fileno())

def atomic_json(path, payload):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
                   encoding="utf-8")
    os.replace(tmp, p)

def atomic_jsonl(path, records):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + ".tmp")
    with tmp.open("w", encoding="utf-8") as fh:
        for record in records:
            fh.write(json.dumps(
                record, ensure_ascii=False, sort_keys=True) + "\n")
        fh.flush()
        os.fsync(fh.fileno())
    os.replace(tmp, p)

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())

# Stale modules are a common source of false provenance.
if any(name == "cre" or name.startswith("cre.f1") for name in sys.modules):
    raise RuntimeError("Stale CRE modules loaded. Restart the runtime, then rerun Cell 1.")

if REPO.exists():
    assert (REPO / ".git").exists(), f"{REPO} exists but is not a git checkout"
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    assert not dirty, f"Refusing to alter a dirty Colab checkout:\n{dirty}"
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
remote_commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", f"origin/{BRANCH}"], text=True).strip()
assert remote_commit == EXPECTED_COMMIT, (remote_commit, EXPECTED_COMMIT)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", remote_commit], check=True)

CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
STATUS = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
BLOB = subprocess.check_output([
    "git", "-C", str(REPO), "rev-parse",
    "HEAD:citation_repair_F1_handoff/cre/f1/band_prompts.py",
], text=True).strip()
assert CODE_COMMIT == EXPECTED_COMMIT
assert STATUS == ""
assert BLOB == EXPECTED_BAND_PROMPTS_BLOB

if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
importlib.invalidate_caches()

from cre.f1 import (
    band_prompts, coverage_aggregate, coverage_prompts_v3, evidence_reader,
    eval_report, fulltext_reader, judgment_run, marker_scope, ncbi_meta, parser,
    preband_contract, preband_disposition, production_launcher, ratelimit,
)
from cre.f1.recording_adapter import AdapterReceipt, wrap_run_seams
import requests

NCBI_API_KEY = get_secret("NCBI_API_KEY")
ratelimit.configure_ncbi(bool(NCBI_API_KEY))

DATA = Path("/content/drive/MyDrive/Citation-Integrity/Data")
HUNT_ROOT = DATA / "mass_error_hunt_soak_v2"
HUNT_ROOT.mkdir(parents=True, exist_ok=True)

def update_attempt_state(status, **extra):
    if "ATTEMPT_STATE_PATH" not in globals():
        return
    previous = {}
    if ATTEMPT_STATE_PATH.exists():
        previous = json.loads(ATTEMPT_STATE_PATH.read_text(encoding="utf-8"))
    previous.update(extra)
    previous.update({"status": status, "updated_at": utc_now()})
    atomic_json(ATTEMPT_STATE_PATH, previous)

def record_crash(stage, exc, *, current_pmcid=None, citation_id=None):
    event = {
        "ts": utc_now(), "stage": stage, "exception_type": type(exc).__name__,
        "message": str(exc), "current_pmcid": current_pmcid,
        "citation_id": citation_id, "traceback": traceback.format_exc(),
        "chunk_index": globals().get("CHUNK_INDEX"),
        "attempt_index": globals().get("ATTEMPT_INDEX"),
    }
    append_jsonl(HUNT_ROOT / "all_crash_events.jsonl", event)
    if "RUN_ROOT" in globals():
        append_jsonl(RUN_ROOT / "crash_events.jsonl", event)
        update_attempt_state("failed", failed_stage=stage,
                             current_pmcid=current_pmcid, citation_id=citation_id)
    return event

print("HEAD:", CODE_COMMIT)
print("git status: CLEAN")
print("band_prompts.py blob:", BLOB)
print("governing modules:", len(production_launcher.GOVERNING_MODULES))
print("cocitation.py launcher-governed:",
      "cocitation.py" in production_launcher.GOVERNING_MODULES)
print("fulltext_reader.py launcher-governed:",
      "fulltext_reader.py" in production_launcher.GOVERNING_MODULES)
print("hunt root:", HUNT_ROOT)
print("BOOTSTRAP: PASS")

## Cell 2 — complete offline suite

In [ ]:
# Free stage. Any count drift stops the run before network or paid work.
# The suite defines MedCPT as an optional dependency and explicitly tests the
# unavailable-model fallback. Colab may already have transformers and may cache
# the weights, so make that one environmental precondition deterministic without
# changing or skipping any CRE test.
OFFLINE_SHIMS = Path("/tmp/cre_offline_suite_shims")
OFFLINE_SHIMS.mkdir(parents=True, exist_ok=True)
(OFFLINE_SHIMS / "transformers.py").write_text(
    "raise ImportError('optional MedCPT disabled for deterministic offline suite')\n",
    encoding="utf-8",
)
offline_env = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join([str(OFFLINE_SHIMS), str(PKG_ROOT)]),
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
t0 = time.time()
proc = subprocess.run(
    [sys.executable, "-m", "pytest", "cre/f1", "-q"],
    cwd=str(PKG_ROOT), text=True, capture_output=True,
    env=offline_env,
)
combined = proc.stdout + "\n" + proc.stderr
print("\n".join(combined.splitlines()[-12:]))
print(f"suite elapsed: {time.time() - t0:.0f}s")
assert proc.returncode == 0, f"pytest failed with exit code {proc.returncode}"

summary = next((line for line in reversed(combined.splitlines()) if " passed" in line), "")
def summary_count(kind):
    hit = re.search(rf"(\d+)\s+{kind}", summary)
    return int(hit.group(1)) if hit else 0

observed = (summary_count("passed"), summary_count("skipped"), summary_count("xfailed"))
EXPECTED_SUITE = (2480, 12, 39)
assert observed == EXPECTED_SUITE, f"suite drift: got {observed}, expected {EXPECTED_SUITE}"
print("OFFLINE SUITE: PASS", observed)

## Cell 3 — deliberately heterogeneous corpus and preserved attempts

The corpus is selected deterministically from multiple eras and document types, then globally interleaved. It is intentionally biased toward difficult shapes and stamped NOT A SEED. The corpus filename, run root, and chunk plan are bound to the strategy version, deterministic seed, and PMCID-list hash.

Set CHUNK_INDEX to choose a chunk. An interrupted paid attempt is never overwritten; rerunning this cell creates a new attempt directory automatically.

In [ ]:
# Free stage.
TOTAL_PAPERS = 200
CHUNK_SIZE = 3
CHUNK_INDEX = 0                         # change this between chunks
CORPUS_SEED = 20260818
STRATEGY_VERSION = "heterogeneous_soak_v3"
# Real PMC documents already known to exercise author-year markup, sentence
# partition diagnostics, co-citation fanout, and a very large bibliography.
MANDATORY_PMCIDS = [
    "PMC12967000", "PMC13219232", "PMC13295838",
    "PMC13294812", "PMC13295119",
]

STRATA = [
    ("era_2000_2007", 30, '"open access"[filter] AND 2000:2007[pdat]'),
    ("era_2008_2013", 30, '"open access"[filter] AND 2008:2013[pdat]'),
    ("era_2014_2019", 30, '"open access"[filter] AND 2014:2019[pdat]'),
    ("era_2020_2025", 30, '"open access"[filter] AND 2020:2025[pdat]'),
    ("reviews", 25, '"open access"[filter] AND "review"[pt]'),
    ("non_english", 20, '"open access"[filter] NOT english[la]'),
    ("retracted_publications", 15, '"open access"[filter] AND "retracted publication"[pt]'),
    ("corrections_errata", 20,
     '"open access"[filter] AND ("published erratum"[pt] OR "corrected and republished article"[pt])'),
]
assert sum(quota for _, quota, _ in STRATA) == TOTAL_PAPERS

PMCID_LIST_PATH = HUNT_ROOT / (
    f"corpus_{STRATEGY_VERSION}_seed{CORPUS_SEED}_n{TOTAL_PAPERS}.json")

def deterministic_key(namespace, pmcid):
    return hashlib.sha256(
        f"{CORPUS_SEED}:{namespace}:{pmcid}".encode("utf-8")).hexdigest()

def esearch(term, retmax):
    params = {
        "db": "pmc", "term": term, "retmax": retmax, "retstart": 0,
        "retmode": "json", "tool": "CRE_mass_error_hunt", "email": EMAIL,
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    response = ratelimit.request_with_retry(
        requests, "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params, limiter=ratelimit.NCBI, timeout=90, max_retries=4,
    )
    if response is None or response.status_code != 200:
        status = None if response is None else response.status_code
        raise RuntimeError(f"ESearch failed with HTTP {status}: {term}")
    payload = response.json()["esearchresult"]
    return [f"PMC{x}" for x in payload.get("idlist", [])], payload

if PMCID_LIST_PATH.exists():
    drawn = json.loads(PMCID_LIST_PATH.read_text(encoding="utf-8"))
    assert drawn["strategy_version"] == STRATEGY_VERSION
    assert drawn["corpus_seed"] == CORPUS_SEED
    assert drawn["requested_total"] == TOTAL_PAPERS
    ids = list(drawn["pmcids"])
    assert len(ids) == TOTAL_PAPERS and len(ids) == len(set(ids))
    assert set(MANDATORY_PMCIDS) <= set(ids)
    assert drawn["pmcid_list_sha256"] == sha256_bytes("\n".join(ids).encode())
    print("reusing persisted heterogeneous corpus:", PMCID_LIST_PATH.name)
else:
    selected = []
    selected_ids = set()
    query_reports = []
    shortages = []

    for name, quota, term in STRATA:
        pool, payload = esearch(term, min(10000, max(1000, quota * 50)))
        ordered = sorted(set(pool), key=lambda p: deterministic_key(name, p))
        chosen = [p for p in ordered if p not in selected_ids][:quota]
        selected.extend({"pmcid": p, "stratum": name} for p in chosen)
        selected_ids.update(chosen)
        if len(chosen) < quota:
            shortages.append({"stratum": name, "wanted": quota, "found": len(chosen)})
        query_reports.append({
            "stratum": name, "term": term, "quota": quota, "pool_ids": len(set(pool)),
            "selected": len(chosen), "available": payload.get("count"),
            "querytranslation": payload.get("querytranslation", ""),
        })

    need = TOTAL_PAPERS - len(selected)
    if need:
        fill_term = '"open access"[filter] AND 1990:2025[pdat]'
        pool, payload = esearch(fill_term, 10000)
        ordered = sorted(set(pool), key=lambda p: deterministic_key("diversity_fill", p))
        fill = [p for p in ordered if p not in selected_ids][:need]
        selected.extend({"pmcid": p, "stratum": "diversity_fill"} for p in fill)
        selected_ids.update(fill)
        query_reports.append({
            "stratum": "diversity_fill", "term": fill_term, "quota": need,
            "pool_ids": len(set(pool)), "selected": len(fill),
            "available": payload.get("count"),
            "querytranslation": payload.get("querytranslation", ""),
        })

    assert len(selected) == TOTAL_PAPERS, (len(selected), shortages)
    # Replace deterministic tail records with the known difficult shapes. This is
    # deliberate adversarial inclusion, not a hidden seed mutation.
    for pmcid in MANDATORY_PMCIDS:
        if pmcid in selected_ids:
            for row in selected:
                if row["pmcid"] == pmcid:
                    row["stratum"] = row["stratum"] + "+known_regression_shape"
            continue
        victim = next(
            row for row in reversed(selected)
            if row["pmcid"] not in MANDATORY_PMCIDS)
        selected.remove(victim)
        selected_ids.remove(victim["pmcid"])
        selected.append({"pmcid": pmcid, "stratum": "known_regression_shape"})
        selected_ids.add(pmcid)
    assert len(selected) == TOTAL_PAPERS
    assert set(MANDATORY_PMCIDS) <= selected_ids
    selected.sort(key=lambda row: deterministic_key("global_interleave", row["pmcid"]))
    ids = [row["pmcid"] for row in selected]
    drawn = {
        "draw_kind": "ADVERSARIAL_HETEROGENEOUS_SOAK_NOT_A_SEED",
        "not_reportable": True,
        "strategy_version": STRATEGY_VERSION,
        "corpus_seed": CORPUS_SEED,
        "requested_total": TOTAL_PAPERS,
        "mandatory_pmcids": MANDATORY_PMCIDS,
        "selected_at": utc_now(),
        "strata": [{"name": n, "quota": q, "term": t} for n, q, t in STRATA],
        "shortages_filled": shortages,
        "query_reports": query_reports,
        "records": selected,
        "pmcids": ids,
        "pmcid_list_sha256": sha256_bytes("\n".join(ids).encode()),
        "note": (
            "Deliberately adversarial engineering corpus. Diagnostic rates describe "
            "this run only and are not population estimates, precision, or prevalence."
        ),
    }
    atomic_json(PMCID_LIST_PATH, drawn)
    print("NEW heterogeneous corpus persisted:", PMCID_LIST_PATH.name)

ALL_PMCIDS = list(drawn["pmcids"])
PMCID_META = {r["pmcid"]: r for r in drawn.get("records", [])}
n_chunks = (len(ALL_PMCIDS) + CHUNK_SIZE - 1) // CHUNK_SIZE
assert 0 <= CHUNK_INDEX < n_chunks, f"CHUNK_INDEX must be 0..{n_chunks - 1}"
CHUNK_PMCIDS = ALL_PMCIDS[
    CHUNK_INDEX * CHUNK_SIZE:(CHUNK_INDEX + 1) * CHUNK_SIZE]

DRAW_ROOT = HUNT_ROOT / (
    f"{STRATEGY_VERSION}_seed{CORPUS_SEED}_{drawn['pmcid_list_sha256'][:16]}")
CHUNK_ROOT = DRAW_ROOT / f"chunk_{CHUNK_INDEX:03d}"
CHUNK_ROOT.mkdir(parents=True, exist_ok=True)

attempts = sorted(CHUNK_ROOT.glob("attempt_*"))
if attempts:
    latest = attempts[-1]
    state_path = latest / "attempt_state.json"
    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}
    unusable = state.get("status") in {"failed", "band1_running", "band2_running"}
    if not state and any(latest.iterdir()):
        unusable = True
    ATTEMPT_INDEX = int(latest.name.split("_")[-1]) + (1 if unusable else 0)
else:
    ATTEMPT_INDEX = 0

RUN_ROOT = CHUNK_ROOT / f"attempt_{ATTEMPT_INDEX:03d}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
ATTEMPT_STATE_PATH = RUN_ROOT / "attempt_state.json"
CHUNK_PLAN_PATH = RUN_ROOT / "chunk_plan.json"
plan = {
    "strategy_version": STRATEGY_VERSION,
    "corpus_seed": CORPUS_SEED,
    "pmcid_list_sha256": drawn["pmcid_list_sha256"],
    "chunk_index": CHUNK_INDEX,
    "attempt_index": ATTEMPT_INDEX,
    "pmcids": CHUNK_PMCIDS,
    "strata": {p: PMCID_META.get(p, {}).get("stratum", "unknown") for p in CHUNK_PMCIDS},
}
if CHUNK_PLAN_PATH.exists():
    assert json.loads(CHUNK_PLAN_PATH.read_text(encoding="utf-8")) == plan
else:
    atomic_json(CHUNK_PLAN_PATH, plan)
if not ATTEMPT_STATE_PATH.exists():
    atomic_json(ATTEMPT_STATE_PATH, {**plan, "status": "planned", "updated_at": utc_now()})

print("draw:", drawn["draw_kind"])
print("corpus hash:", drawn["pmcid_list_sha256"])
print(f"papers: {len(ALL_PMCIDS)} | chunks: {n_chunks} | chunk size: {CHUNK_SIZE}")
print("chunk:", CHUNK_INDEX, CHUNK_PMCIDS)
print("strata:", plan["strata"])
print("attempt:", ATTEMPT_INDEX, RUN_ROOT)

## Cell 4 — retrieve, validate, profile, and parse every input paper

In [ ]:
# Free stage. Every HTTP attempt and every per-paper parser outcome is durable.
class ObservedSession:
    def __init__(self, base, events_path, stage):
        self.base = base
        self.events_path = Path(events_path)
        self.stage = stage

    def get(self, url, **kwargs):
        started = time.time()
        params = kwargs.get("params") or {}
        safe_params = {
            str(k): str(v)[:500] for k, v in params.items()
            if str(k).lower() != "api_key"
        }
        event = {
            "ts": utc_now(), "stage": self.stage,
            "host": urlparse(url).netloc, "path": urlparse(url).path,
            "params": safe_params,
        }
        try:
            response = self.base.get(url, **kwargs)
            event.update({
                "result": "http_response", "status_code": response.status_code,
                "elapsed_s": round(time.time() - started, 3),
                "response_bytes": len(getattr(response, "content", b"") or b""),
            })
            append_jsonl(self.events_path, event)
            return response
        except Exception as exc:
            event.update({
                "result": "transport_exception", "exception_type": type(exc).__name__,
                "message": str(exc), "elapsed_s": round(time.time() - started, 3),
            })
            append_jsonl(self.events_path, event)
            raise

TRANSPORT_EVENTS_PATH = RUN_ROOT / "transport_events.jsonl"
INPUT_PROFILE_PATH = RUN_ROOT / "input_profile.json"
CANDIDATE_XML_DIR = RUN_ROOT / "candidate_xml"
XML_DIR = RUN_ROOT / "active_corpus"
DOWNLOAD_DIR = DRAW_ROOT / "retrieved_papers"
for directory in (CANDIDATE_XML_DIR, XML_DIR, DOWNLOAD_DIR):
    directory.mkdir(parents=True, exist_ok=True)

observed_retrieval_session = ObservedSession(
    requests.Session(), TRANSPORT_EVENTS_PATH, "citing_pmc_efetch")

def validate_jats(payload):
    if len(payload) < 1000:
        return None, f"implausibly_small_xml:{len(payload)}"
    try:
        root = ET.fromstring(payload)
    except Exception as exc:
        return None, f"xml_parse:{type(exc).__name__}:{exc}"
    local = root.tag.rsplit("}", 1)[-1]
    articles = [node for node in root.iter()
                if isinstance(node.tag, str) and node.tag.rsplit("}", 1)[-1] == "article"]
    if local != "article" and not articles:
        return None, "no_jats_article"
    return root, ""

def retrieve_pmc_xml(pmcid):
    assert re.fullmatch(r"PMC\d+", pmcid), pmcid
    cache = DOWNLOAD_DIR / f"{pmcid}.xml"
    if cache.exists():
        payload = cache.read_bytes()
        root, error = validate_jats(payload)
        if root is not None:
            return payload, root, "cache"
        append_jsonl(TRANSPORT_EVENTS_PATH, {
            "ts": utc_now(), "stage": "citing_pmc_cache",
            "pmcid": pmcid, "result": "cached_invalid", "detail": error,
        })

    params = {
        "db": "pmc", "id": pmcid[3:], "retmode": "xml",
        "tool": "CRE_mass_error_hunt", "email": EMAIL,
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    response = ratelimit.request_with_retry(
        observed_retrieval_session,
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
        params, limiter=ratelimit.NCBI, timeout=90, max_retries=4,
    )
    if response is None or response.status_code != 200:
        status = None if response is None else response.status_code
        raise RuntimeError(f"PMC EFetch final status {status}")
    payload = response.content
    root, error = validate_jats(payload)
    if root is None:
        raise ValueError(error)
    if not cache.exists():
        cache.write_bytes(payload)
    return payload, root, "live"

INPUT_PROFILES = {}
PARSED_REFS_BY_DOC = {}
EXPECTED_CITATION_IDS = set()

for pmcid in CHUNK_PMCIDS:
    profile = {
        "pmcid": pmcid,
        "selection_stratum": PMCID_META.get(pmcid, {}).get("stratum", "unknown"),
        "retrieval_status": None, "parse_status": None, "reference_count": 0,
        "missing_pmid": 0, "missing_citance": 0, "corporate_first_author": 0,
        "non_ascii_titles": 0, "sentence_partition_events": 0,
        "sentence_partition_uncovered_characters": 0,
        "citation_style": "unknown", "language": "",
    }
    try:
        payload, root, source = retrieve_pmc_xml(pmcid)
        profile["retrieval_status"] = source
        profile["xml_bytes"] = len(payload)
        article = root if root.tag.rsplit("}", 1)[-1] == "article" else next(
            node for node in root.iter()
            if isinstance(node.tag, str) and node.tag.rsplit("}", 1)[-1] == "article")
        profile["language"] = (
            article.attrib.get("{http://www.w3.org/XML/1998/namespace}lang")
            or article.attrib.get("lang") or ""
        )
        candidate_path = CANDIDATE_XML_DIR / f"{pmcid}.xml"
        if candidate_path.exists():
            assert candidate_path.read_bytes() == payload
        else:
            candidate_path.write_bytes(payload)

        refs = parser.parse_pmc_xml(str(candidate_path), source_pmcid=pmcid)
        profile["parse_status"] = "parsed"
        profile["reference_count"] = len(refs)
        profile["missing_pmid"] = sum(not r.claimed.claimed_pmid for r in refs)
        profile["missing_citance"] = sum(not r.citance for r in refs)
        profile["corporate_first_author"] = sum(
            bool(r.claimed.first_author_is_collab) for r in refs)
        profile["non_ascii_titles"] = sum(
            any(ord(ch) > 127 for ch in (r.claimed.title or "")) for r in refs)
        failures = [
            f for ref in refs
            for f in (ref.citance_sentence_partition_failures or [])
        ]
        profile["sentence_partition_events"] = len(failures)
        profile["sentence_partition_uncovered_characters"] = sum(
            int(f.get("uncovered_chars") or 0) for f in failures)
        profile["citation_style"] = marker_scope.detect_citation_style(
            [r.cited_reference_marker for r in refs])
        if not refs:
            profile["input_failure"] = "zero_references"

        active_path = XML_DIR / candidate_path.name
        if active_path.exists():
            assert active_path.read_bytes() == payload
        else:
            shutil.copy2(candidate_path, active_path)
        PARSED_REFS_BY_DOC[pmcid] = refs
        for ref in refs:
            if ref.citation_id in EXPECTED_CITATION_IDS:
                raise RuntimeError(f"duplicate citation_id across corpus: {ref.citation_id}")
            EXPECTED_CITATION_IDS.add(ref.citation_id)
    except Exception as exc:
        profile["parse_status"] = "failed"
        profile["input_failure"] = f"{type(exc).__name__}: {exc}"
        append_jsonl(RUN_ROOT / "input_failures.jsonl", {
            "ts": utc_now(), "pmcid": pmcid, "stage": "retrieve_or_parse",
            "exception_type": type(exc).__name__, "message": str(exc),
            "traceback": traceback.format_exc(),
        })
    INPUT_PROFILES[pmcid] = profile

atomic_json(INPUT_PROFILE_PATH, {
    "chunk_index": CHUNK_INDEX, "attempt_index": ATTEMPT_INDEX,
    "requested_papers": CHUNK_PMCIDS, "profiles": INPUT_PROFILES,
})

CORPUS_MANIFEST_PATH = RUN_ROOT / "active_corpus_manifest.json"
if CORPUS_MANIFEST_PATH.exists():
    cm = json.loads(CORPUS_MANIFEST_PATH.read_text(encoding="utf-8"))
    preband_contract.verify_corpus_contents(
        str(XML_DIR), preband_contract.corpus_inventory(cm))
else:
    cm = preband_contract.build_corpus_manifest(
        str(XML_DIR), str(CORPUS_MANIFEST_PATH))

REFERENCE_COUNTS = {
    pmcid: profile["reference_count"] for pmcid, profile in INPUT_PROFILES.items()
    if profile["parse_status"] == "parsed"
}
CHUNK_REFS = sum(REFERENCE_COUNTS.values())
zero_ref_docs = sorted(p for p, n in REFERENCE_COUNTS.items() if n == 0)
failed_docs = sorted(p for p, row in INPUT_PROFILES.items()
                     if row["parse_status"] != "parsed")

update_attempt_state(
    "setup_complete", active_papers=len(REFERENCE_COUNTS),
    references=CHUNK_REFS, zero_reference_papers=zero_ref_docs,
    input_failed_papers=failed_docs,
)

print("INPUT PROFILE COMES BEFORE THE PAID GATE")
for pmcid in CHUNK_PMCIDS:
    print(pmcid, json.dumps(INPUT_PROFILES[pmcid], ensure_ascii=False, sort_keys=True))
print(f"\nrequested papers: {len(CHUNK_PMCIDS)}")
print(f"parser-success papers: {len(REFERENCE_COUNTS)}")
print(f"input failures: {len(failed_docs)} {failed_docs}")
print(f"zero-reference papers: {len(zero_ref_docs)} {zero_ref_docs}")
print(f"parsed references: {CHUNK_REFS}")
print(f"unique citation ids: {len(EXPECTED_CITATION_IDS)}")
assert CHUNK_REFS > 0, (
    "This chunk contains no references after input profiling. The failure is durable; "
    "do not enter the paid stages for this chunk.")
print("RETRIEVE, PROFILE, AND PARSE: PASS")

## Cell 5 — cost gate based on observed calls and tokens, never a paper-count guess

In [ ]:
# Hard stop before the first paid call.
ENABLE_PAID_RUN = False
BAND2_MAX_TOKENS = 2048
ANTHROPIC_MAX_RETRIES = 3
PROMPT_CACHE_TTL = "5m"
PROMPT_CACHE_SUPPORTED_TTLS = {"5m", "1h"}

def stable_prefix_before_slot(template, slot):
    if not isinstance(template, str) or not template:
        raise ValueError("prompt template must be nonempty text")
    if not isinstance(slot, str) or not slot or template.count(slot) != 1:
        raise ValueError("prompt cache slot must occur exactly once")
    prefix = template.partition(slot)[0]
    if not prefix:
        raise ValueError("prompt cache prefix must not be empty")
    return prefix

def join_text_content(content):
    if isinstance(content, str):
        return content
    if not isinstance(content, list):
        raise ValueError("message content must be text or a list of text blocks")
    if any(not isinstance(block, dict) or block.get("type") != "text"
           or not isinstance(block.get("text"), str) for block in content):
        raise ValueError("cached message content may contain only text blocks")
    return "".join(block["text"] for block in content)

def cached_text_content(prompt, stable_prefix, ttl):
    if ttl not in PROMPT_CACHE_SUPPORTED_TTLS:
        raise ValueError(f"unsupported prompt-cache TTL {ttl!r}")
    if not isinstance(prompt, str) or not prompt.startswith(stable_prefix):
        raise ValueError("rendered prompt does not begin with the configured cache prefix")
    content = [
        {"type": "text", "text": stable_prefix,
         "cache_control": {"type": "ephemeral", "ttl": ttl}},
        {"type": "text", "text": prompt[len(stable_prefix):]},
    ]
    if join_text_content(content) != prompt:
        raise AssertionError("prompt-cache blocks changed the model input")
    return content

FULLTEXT_COVERAGE_CACHE_PREFIX = stable_prefix_before_slot(
    coverage_prompts_v3.COVERAGE_PROMPT_V3, "<<ATOMIC_CLAIM>>")
_cache_probe = FULLTEXT_COVERAGE_CACHE_PREFIX + "quality-neutral cache probe"
assert join_text_content(cached_text_content(
    _cache_probe, FULLTEXT_COVERAGE_CACHE_PREFIX, PROMPT_CACHE_TTL)) == _cache_probe

OPUS5_PRICE_USD_PER_MTOK = {
    "input_tokens": 5.00,
    "cache_creation_input_tokens": 6.25,
    "cache_read_input_tokens": 0.50,
    "output_tokens": 25.00,
}
OPUS5_PRICE_SOURCE = "https://platform.claude.com/docs/en/about-claude/pricing"

def opus5_cost_summary(events):
    totals = {key: sum(int(e.get(key) or 0) for e in events)
              for key in OPUS5_PRICE_USD_PER_MTOK}
    billed = sum(totals[key] * price
                 for key, price in OPUS5_PRICE_USD_PER_MTOK.items()) / 1_000_000
    all_input = (totals["input_tokens"] +
                 totals["cache_creation_input_tokens"] +
                 totals["cache_read_input_tokens"])
    uncached = (all_input * OPUS5_PRICE_USD_PER_MTOK["input_tokens"] +
                totals["output_tokens"] * OPUS5_PRICE_USD_PER_MTOK["output_tokens"]) / 1_000_000
    return {
        **totals,
        "estimated_billed_usd": round(billed, 6),
        "uncached_equivalent_usd": round(uncached, 6),
        "estimated_cache_savings_usd": round(max(0.0, uncached - billed), 6),
        "pricing_source": OPUS5_PRICE_SOURCE,
    }

completed_censuses = sorted(DRAW_ROOT.glob("chunk_*/attempt_*/chunk_census.json"))
observed_model_events = []
for census_path in completed_censuses:
    events_path = census_path.parent / "model_call_events.jsonl"
    observed_model_events.extend(read_jsonl(events_path))

successful_calls = [e for e in observed_model_events
                    if e.get("result") == "success" and e.get("model") == MODEL]
input_tokens = sum(int(e.get("input_tokens") or 0) for e in successful_calls)
output_tokens = sum(int(e.get("output_tokens") or 0) for e in successful_calls)
observed_cost = opus5_cost_summary(successful_calls)

print("COST PREFLIGHT")
print("This notebook does not infer dollars from paper or reference counts.")
print("Current chunk:")
print("  papers requested:", len(CHUNK_PMCIDS))
print("  parser-success papers:", len(REFERENCE_COUNTS))
print("  parsed references:", CHUNK_REFS)
print("  selection strata:", {p: PMCID_META.get(p, {}).get("stratum") for p in CHUNK_PMCIDS})
if successful_calls:
    print("Observed across completed chunks:")
    print("  provider calls:", len(successful_calls))
    print("  input tokens:", input_tokens)
    print("  output tokens:", output_tokens)
    print("  cache-write tokens:", observed_cost["cache_creation_input_tokens"])
    print("  cache-read tokens:", observed_cost["cache_read_input_tokens"])
    print("  estimated billed USD:", observed_cost["estimated_billed_usd"])
    print("  estimated cache savings USD:", observed_cost["estimated_cache_savings_usd"])
    print("Use the Anthropic console for actual billed cost; token totals are retained for reconciliation.")
else:
    print("No completed paid chunk exists. Cost is UNKNOWN until the first repaired pilot finishes.")
print("Cost mode: same Opus 5 prompts with an explicit 5-minute cache on the invariant full-text coverage prefix.")
print("Full-text coverage still uses one provider request per atomic claim; parsed references are not a cost unit.")

if ENABLE_PAID_RUN is not True:
    raise RuntimeError("COST GATE CLOSED — set ENABLE_PAID_RUN = True only after reading this cell")

## Cell 6 — Unicode-safe live watcher

In [ ]:
# Starts a daemon watcher. It reads bytes so Unicode cannot corrupt file offsets.
if "_watch_stop" in globals():
    _watch_stop.set()
if "_live_thread" in globals() and _live_thread.is_alive():
    _live_thread.join(timeout=2)

LIVE = {
    "pairs": 0, "positive": 0, "quarantined": 0,
    "labels": collections.Counter(), "docs": set(), "t0": time.time(), "hits": [],
}
_watch_stop = threading.Event()
BAND2_OUT = RUN_ROOT / "band2"

def labels_of(record):
    labels = list(record.get("findings") or [])
    if any((s or {}).get("derived") == "F4"
           for s in (record.get("strength_records") or [])):
        labels.append("F4")
    label = record.get("label")
    if label:
        labels.append(label)
    return sorted(set(labels))

def _watch(pred_path, poll=1.0, heartbeat=60.0):
    position = 0
    last_heartbeat = time.time()
    while not _watch_stop.is_set():
        if os.path.exists(pred_path):
            with open(pred_path, "rb") as fh:
                fh.seek(position)
                while True:
                    raw = fh.readline()
                    if not raw or not raw.endswith(b"\n"):
                        break
                    position = fh.tell()
                    try:
                        record = json.loads(raw.decode("utf-8"))
                    except (UnicodeDecodeError, json.JSONDecodeError):
                        continue
                    LIVE["pairs"] += 1
                    if record.get("citing_pmcid"):
                        LIVE["docs"].add(record["citing_pmcid"])
                    if record.get("disposition") == judgment_run.DISP_QUARANTINE_PARSE:
                        LIVE["quarantined"] += 1
                    labels = labels_of(record)
                    if not labels:
                        continue
                    LIVE["positive"] += 1
                    for label in labels:
                        LIVE["labels"][label] += 1
                    sentence = (record.get("citing_sentence") or "").replace("\n", " ")
                    LIVE["hits"].append({
                        "citation_id": record.get("citation_id"), "labels": labels,
                        "cited_pmid": record.get("cited_pmid"),
                        "disposition": record.get("disposition"),
                        "citing_sentence": sentence,
                    })
                    print(f"\n>>> [{'+'.join(labels)}] {record.get('citation_id')} "
                          f"disposition={record.get('disposition')}", flush=True)
                    print("    " + sentence[:240], flush=True)
        now = time.time()
        if now - last_heartbeat >= heartbeat:
            last_heartbeat = now
            rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            print(f"[live] docs={len(LIVE['docs'])} pairs={LIVE['pairs']} "
                  f"positive={LIVE['positive']} quarantined={LIVE['quarantined']} "
                  f"peak_rss_kb={rss} elapsed_min={(now-LIVE['t0'])/60:.1f}",
                  flush=True)
        _watch_stop.wait(poll)

def stop_live_watch():
    _watch_stop.set()
    if "_live_thread" in globals() and _live_thread.is_alive():
        _live_thread.join(timeout=3)
    print(f"[live] FINAL docs={len(LIVE['docs'])} pairs={LIVE['pairs']} "
          f"positive={LIVE['positive']} quarantined={LIVE['quarantined']} "
          f"labels={dict(LIVE['labels'])}")

_live_thread = threading.Thread(
    target=_watch,
    args=(str(BAND2_OUT / "judgment_predictions.jsonl"),),
    daemon=True,
)
_live_thread.start()
print("[live] watching", BAND2_OUT / "judgment_predictions.jsonl")

## Cell 7 — Band 1 with retries, complete-domain validation, and crash logging

In [ ]:
from unittest.mock import patch
from cre.f1 import run as band1_run

assert ENABLE_PAID_RUN is True
ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY", required=True)

BAND1_DIR = RUN_ROOT / "band1"
BAND1_DIR.mkdir(parents=True, exist_ok=True)
BAND1_PREDICTIONS = BAND1_DIR / "band1_predictions.jsonl"
BAND1_LOGS = BAND1_DIR / "band1_logs.jsonl"
CURRENT_CONTEXT = {"pmcid": None, "citation_id": None}

def tracked_refs():
    for pmcid in sorted(PARSED_REFS_BY_DOC):
        for ref in PARSED_REFS_BY_DOC[pmcid]:
            CURRENT_CONTEXT["pmcid"] = pmcid
            CURRENT_CONTEXT["citation_id"] = ref.citation_id
            yield ref

if BAND1_LOGS.exists() and BAND1_PREDICTIONS.exists():
    band1_logs = read_jsonl(BAND1_LOGS)
    print("Reusing validated Band-1 artifacts.")
    BAND1_MODEL_CALLS = None
else:
    assert not BAND1_LOGS.exists() and not BAND1_PREDICTIONS.exists(), (
        "Partial Band-1 artifacts are preserved. Rerun Cell 3 to start a new attempt.")
    update_attempt_state("band1_running")
    from anthropic import Anthropic
    MODEL_CALL_EVENTS_PATH = RUN_ROOT / "model_call_events.jsonl"
    band1_client = Anthropic(
        api_key=ANTHROPIC_API_KEY, max_retries=ANTHROPIC_MAX_RETRIES,
        timeout=120.0)
    BAND1_MODEL_CALLS = 0

    def counted_complete(prompt):
        global BAND1_MODEL_CALLS
        BAND1_MODEL_CALLS += 1
        started = time.time()
        event = {
            "ts": utc_now(), "stage": "band1_llm", "model": MODEL,
            "prompt_chars": len(prompt), "max_tokens": 400,
        }
        if BAND1_MODEL_CALLS == 1 or BAND1_MODEL_CALLS % 25 == 0:
            print(f"[band1] model calls: {BAND1_MODEL_CALLS}", flush=True)
        try:
            response = band1_client.messages.create(
                model=MODEL, max_tokens=400,
                messages=[{"role": "user", "content": prompt}])
            text = band1_run._extract_text(response)
            usage = getattr(response, "usage", None)
            event.update({
                "result": "success",
                "elapsed_s": round(time.time() - started, 3),
                "output_chars": len(text),
                "output_sha256": sha256_bytes(text.encode("utf-8")),
                "input_tokens": int(getattr(usage, "input_tokens", 0) or 0),
                "output_tokens": int(getattr(usage, "output_tokens", 0) or 0),
                "cache_creation_input_tokens": int(
                    getattr(usage, "cache_creation_input_tokens", 0) or 0),
                "cache_read_input_tokens": int(
                    getattr(usage, "cache_read_input_tokens", 0) or 0),
            })
            append_jsonl(MODEL_CALL_EVENTS_PATH, event)
            return text
        except Exception as exc:
            event.update({
                "result": "provider_exception",
                "exception_type": type(exc).__name__, "message": str(exc),
                "status_code": getattr(exc, "status_code", None),
                "elapsed_s": round(time.time() - started, 3),
            })
            append_jsonl(MODEL_CALL_EVENTS_PATH, event)
            if band1_run._is_retryable(exc):
                return ""
            raise band1_run.NonRetryableProviderError(str(exc)) from exc

    observed_band1_session = ObservedSession(
        requests.Session(), TRANSPORT_EVENTS_PATH, "band1_network")
    try:
        with patch.object(band1_run, "make_completer", return_value=counted_complete), \
             patch.object(band1_run.requests, "Session",
                          return_value=observed_band1_session):
            BAND1_COUNTS = band1_run.run(
                str(XML_DIR), str(BAND1_PREDICTIONS), str(BAND1_LOGS),
                model=MODEL, anthropic_key=ANTHROPIC_API_KEY,
                ncbi_key=NCBI_API_KEY, crossref_mailto=EMAIL,
                openalex_mailto=EMAIL, refs=tracked_refs())
        band1_logs = read_jsonl(BAND1_LOGS)
    except Exception as exc:
        record_crash(
            "band1", exc, current_pmcid=CURRENT_CONTEXT["pmcid"],
            citation_id=CURRENT_CONTEXT["citation_id"])
        stop_live_watch()
        raise
    finally:
        CURRENT_CONTEXT.update(pmcid=None, citation_id=None)

band1_ids = [row.get("citation_id") for row in band1_logs]
assert len(band1_ids) == len(set(band1_ids))
assert set(band1_ids) == EXPECTED_CITATION_IDS, {
    "missing": sorted(EXPECTED_CITATION_IDS - set(band1_ids))[:10],
    "extra": sorted(set(band1_ids) - EXPECTED_CITATION_IDS)[:10],
}
BAND1_COUNTS = dict(collections.Counter(row.get("label") for row in band1_logs))
BAND1_REPORT = eval_report.summarize(band1_logs)
f1_status = BAND1_REPORT["f1_status"]
resolved_rows = [r for r in band1_logs if (r.get("log") or {}).get("pmid_resolved")]
f8_precursor_answered = sum(
    (r.get("log") or {}).get("retracted") is not None for r in resolved_rows)
snapshot = datetime.now(timezone.utc).date().isoformat()

check_attestations = {
    "F1": {
        "performed": True, "source": "Band-1 existence check",
        "snapshot_date": snapshot, "attempted": f1_status["attempted"],
        "answered": f1_status["answered"],
        "transport_failed": f1_status["transport_failed"],
        "fired": BAND1_COUNTS.get("F1", 0),
    },
    "F2": {
        "performed": True, "source": "Band-1 identity matcher",
        "snapshot_date": snapshot, "attempted": len(band1_logs),
        "answered": len(band1_logs) - f1_status["transport_failed"],
        "transport_failed": f1_status["transport_failed"],
        "fired": BAND1_COUNTS.get("F2", 0),
    },
    "F8": {
        "performed": True,
        "source": "PubMed Retracted Publication precursor only",
        "taxonomy_implemented": False,
        "timing_rule_implemented": False,
        "snapshot_date": snapshot, "attempted": len(resolved_rows),
        "answered": f8_precursor_answered,
        "transport_failed": len(resolved_rows) - f8_precursor_answered,
        "precursor_fired": BAND1_COUNTS.get("F8", 0),
    },
}

DISPOSITION_PATH = BAND1_DIR / preband_disposition.ARTIFACT_FILENAME
DISPOSITION_MANIFEST_PATH = Path(
    str(DISPOSITION_PATH) + preband_disposition.MANIFEST_SUFFIX)
if DISPOSITION_PATH.exists() or DISPOSITION_MANIFEST_PATH.exists():
    assert DISPOSITION_PATH.exists() and DISPOSITION_MANIFEST_PATH.exists()
    disposition_manifest = json.loads(
        DISPOSITION_MANIFEST_PATH.read_text(encoding="utf-8"))
else:
    disposition_manifest = preband_disposition.write_disposition(
        str(BAND1_LOGS), str(DISPOSITION_PATH), f2_commit=CODE_COMMIT,
        corpus_manifest_path=str(CORPUS_MANIFEST_PATH),
        generated_by="CRE heterogeneous soak test (NOT REPORTABLE)",
        generated_at=utc_now(), check_attestations=check_attestations)

assert disposition_manifest["row_count"] == len(band1_logs)
atomic_json(RUN_ROOT / "band1_summary.json", {
    "labels": BAND1_COUNTS, "f1_status": f1_status,
    "model_calls": BAND1_MODEL_CALLS,
    "f8_precursor": check_attestations["F8"],
})
update_attempt_state("band1_complete", band1_rows=len(band1_logs))

print("Band-1 labels:", json.dumps(BAND1_COUNTS, sort_keys=True))
print("F1 transport attempted/answered/failed:",
      f1_status["attempted"], f1_status["answered"], f1_status["transport_failed"])
print("F8 taxonomy detector: NOT IMPLEMENTED")
print("F8 publication-type precursor answered:", f8_precursor_answered)
print("BAND 1: PASS")

## Cell 8 — full-text Band 2 with quality-neutral prompt caching, provider telemetry, retries, and durable crash state

In [ ]:
from anthropic import Anthropic

assert ENABLE_PAID_RUN is True
BAND2_MANIFEST_PATH = BAND2_OUT / "judgment_run_manifest.json"
MODEL_CALL_EVENTS_PATH = RUN_ROOT / "model_call_events.jsonl"
ABSTRACT_EVENTS_PATH = RUN_ROOT / "abstract_evidence_events.jsonl"
FULLTEXT_EVENTS_PATH = RUN_ROOT / "fulltext_evidence_events.jsonl"
RUNTIME_EVENTS_PATH = RUN_ROOT / "runtime_events.jsonl"

def usage_dict(response):
    usage = getattr(response, "usage", None)
    keys = (
        "input_tokens", "output_tokens", "cache_creation_input_tokens",
        "cache_read_input_tokens",
    )
    return {key: int(getattr(usage, key, 0) or 0) for key in keys}

def make_logged_anthropic_call(client, stage, *, cache_prefix=None):
    def call(prompt):
        started = time.time()
        prompt_sha256 = sha256_bytes(prompt.encode("utf-8"))
        if cache_prefix is None:
            message_content = prompt
        else:
            message_content = cached_text_content(
                prompt, cache_prefix, PROMPT_CACHE_TTL)
            assert join_text_content(message_content) == prompt
        event = {
            "ts": utc_now(), "stage": stage, "model": MODEL,
            "prompt_chars": len(prompt), "prompt_sha256": prompt_sha256,
            "max_tokens": BAND2_MAX_TOKENS,
            "prompt_cache_requested": cache_prefix is not None,
            "prompt_cache_ttl": PROMPT_CACHE_TTL if cache_prefix is not None else None,
            "prompt_cache_prefix_chars": len(cache_prefix or ""),
            "prompt_cache_prefix_sha256": (
                sha256_bytes(cache_prefix.encode("utf-8")) if cache_prefix else None),
        }
        try:
            response = client.messages.create(
                model=MODEL, max_tokens=BAND2_MAX_TOKENS,
                messages=[{"role": "user", "content": message_content}],
            )
            text = "".join(
                block.text for block in response.content
                if getattr(block, "type", None) == "text")
            event.update({
                "result": "success", "elapsed_s": round(time.time() - started, 3),
                "output_chars": len(text),
                "output_sha256": sha256_bytes(text.encode("utf-8")),
                **usage_dict(response),
            })
            append_jsonl(MODEL_CALL_EVENTS_PATH, event)
            return text
        except Exception as exc:
            event.update({
                "result": "provider_exception",
                "exception_type": type(exc).__name__, "message": str(exc),
                "status_code": getattr(exc, "status_code", None),
                "elapsed_s": round(time.time() - started, 3),
            })
            append_jsonl(MODEL_CALL_EVENTS_PATH, event)
            raise
    return call

if BAND2_MANIFEST_PATH.exists():
    BAND2_MANIFEST = json.loads(BAND2_MANIFEST_PATH.read_text(encoding="utf-8"))
    assert BAND2_MANIFEST.get("status") == "complete", (
        "Incomplete Band-2 output is preserved. Rerun Cell 3 to create a new attempt.")
    print("Reusing complete Band-2 artifacts.")
    stop_live_watch()
else:
    if BAND2_OUT.exists() and any(BAND2_OUT.iterdir()):
        raise RuntimeError(
            "Partial Band-2 output is preserved. Rerun Cell 3 for a new attempt.")
    BAND2_OUT.mkdir(parents=True, exist_ok=True)
    update_attempt_state("band2_running")
    append_jsonl(RUNTIME_EVENTS_PATH, {
        "ts": utc_now(), "event": "band2_start",
        "peak_rss_kb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
        "papers": len(REFERENCE_COUNTS), "references": CHUNK_REFS,
    })

    client = Anthropic(
        api_key=ANTHROPIC_API_KEY, max_retries=ANTHROPIC_MAX_RETRIES,
        timeout=120.0)
    extract_call = make_logged_anthropic_call(client, "claim_extraction")
    abstract_coverage_call = make_logged_anthropic_call(
        client, "coverage_abstract_fallback")
    fulltext_coverage_call = make_logged_anthropic_call(
        client, "coverage_fulltext_v3",
        cache_prefix=FULLTEXT_COVERAGE_CACHE_PREFIX)
    discriminator_call = make_logged_anthropic_call(client, "f3_f4_discriminator")
    verifier_call = make_logged_anthropic_call(client, "f4_verifier")

    receipt = AdapterReceipt(
        model=MODEL, temperature="unsupported", assistant_prefill="unsupported")
    observed_ncbi_session = ObservedSession(
        requests.Session(), TRANSPORT_EVENTS_PATH, "band2_ncbi")
    abstract_cache = DRAW_ROOT / "abstract_cache"
    fulltext_cache = DRAW_ROOT / "fulltext_cache"

    def fetch_abstract_observed(pmid):
        pmid = str(pmid or "").strip()
        if not pmid:
            append_jsonl(ABSTRACT_EVENTS_PATH, {
                "ts": utc_now(), "pmid": pmid, "result": "unattemptable"})
            return None
        cached = evidence_reader._read_cache(str(abstract_cache), pmid)
        if cached is not None:
            append_jsonl(ABSTRACT_EVENTS_PATH, {
                "ts": utc_now(), "pmid": pmid, "result": "usable_abstract",
                "source": "cache"})
            return cached
        params = {
            "db": "pubmed", "id": pmid, "rettype": "abstract", "retmode": "xml",
            "tool": ncbi_meta.TOOL, "email": EMAIL,
        }
        if NCBI_API_KEY:
            params["api_key"] = NCBI_API_KEY
        try:
            response = ratelimit.request_with_retry(
                observed_ncbi_session, ncbi_meta.EFETCH, params,
                limiter=ratelimit.NCBI, timeout=20, max_retries=4)
        except requests.RequestException as exc:
            append_jsonl(ABSTRACT_EVENTS_PATH, {
                "ts": utc_now(), "pmid": pmid, "result": "transport_error",
                "message": str(exc)})
            return None
        if response is None or response.status_code != 200:
            append_jsonl(ABSTRACT_EVENTS_PATH, {
                "ts": utc_now(), "pmid": pmid, "result": "http_failure",
                "status_code": None if response is None else response.status_code})
            return None
        if not response.text.strip():
            append_jsonl(ABSTRACT_EVENTS_PATH, {
                "ts": utc_now(), "pmid": pmid, "result": "empty_response"})
            return None
        abstract = evidence_reader._parse_abstract(response.text)
        if abstract is None:
            append_jsonl(ABSTRACT_EVENTS_PATH, {
                "ts": utc_now(), "pmid": pmid,
                "result": "answered_no_usable_abstract"})
            return None
        if abstract.strip().casefold() in band_prompts._MISSING_ABSTRACT_SENTINELS:
            append_jsonl(ABSTRACT_EVENTS_PATH, {
                "ts": utc_now(), "pmid": pmid,
                "result": "answered_sentinel_abstract"})
            return None
        evidence_reader._write_cache(str(abstract_cache), pmid, abstract)
        append_jsonl(ABSTRACT_EVENTS_PATH, {
            "ts": utc_now(), "pmid": pmid, "result": "usable_abstract",
            "source": "live"})
        return abstract

    def fetch_fulltext_observed(pmid):
        result = fulltext_reader.fetch_fulltext(
            pmid, api_key=NCBI_API_KEY, email=EMAIL,
            session=observed_ncbi_session, cache_dir=str(fulltext_cache))
        append_jsonl(FULLTEXT_EVENTS_PATH, {
            "ts": utc_now(), "pmid": str(pmid or ""),
            "result": "unattemptable" if result is None else (
                "complete" if result.get("retrieval_complete") is True else "incomplete"),
            "pmcid": None if result is None else result.get("pmcid"),
            "source": None if result is None else result.get("source"),
            "incomplete_reasons": [] if result is None
                else list(result.get("incomplete_reasons") or []),
            "sections_present": [] if result is None
                else list(result.get("sections_present") or []),
        })
        return result

    def fetch_reflist(pmcid):
        return ncbi_meta.ncbi_pmc_reflist(
            pmcid, api_key=NCBI_API_KEY, email=EMAIL,
            session=observed_ncbi_session)

    def resolve_pmcid(pmid):
        return fulltext_reader._live_resolve_pmcid(
            pmid, NCBI_API_KEY, EMAIL, observed_ncbi_session)

    def pubtypes_lookup(pmid):
        return ncbi_meta.ncbi_pubtypes(
            pmid, NCBI_API_KEY, EMAIL, session=observed_ncbi_session)

    seams = wrap_run_seams(
        receipt,
        extractor=band_prompts.make_extractor(extract_call),
        coverage_judge=coverage_aggregate.make_coverage_judge(
            abstract_coverage_call),
        fetch_abstract=fetch_abstract_observed,
        fetch_reflist=fetch_reflist,
        fetch_fulltext=fetch_fulltext_observed,
        coverage_judge_v3=coverage_prompts_v3.make_coverage_judge_v3(
            fulltext_coverage_call),
        discriminator_call_llm=discriminator_call,
        f4_verifier_call_llm=verifier_call,
        f3_fetch_reflist=fetch_reflist,
        f3_resolve_pmcid=resolve_pmcid,
        pubtypes_lookup=pubtypes_lookup,
    )

    print(f"Starting full-text Band 2: chunk={CHUNK_INDEX}, "
          f"papers={len(REFERENCE_COUNTS)}, refs={CHUNK_REFS}", flush=True)
    started = time.time()
    try:
        BAND2_MANIFEST = judgment_run.run_natural_judgment(
            str(XML_DIR), str(BAND2_OUT),
            preband_disposition=str(DISPOSITION_PATH),
            corpus_manifest_path=str(CORPUS_MANIFEST_PATH),
            code_commit=CODE_COMMIT, model=MODEL,
            email=EMAIL, api_key=NCBI_API_KEY, session=observed_ncbi_session,
            f4_verifier_model_id=MODEL,
            assistant_prefill="unsupported", temperature="unsupported",
            require_full_coverage=True, require_reportable=False, production=False,
            **seams,
        )
    except Exception as exc:
        completed = {
            row.get("pmcid") for row in
            read_jsonl(BAND2_OUT / "judgment_run_checkpoint.jsonl")
            if row.get("pmcid")
        }
        current = next(
            (p for p in sorted(REFERENCE_COUNTS) if p not in completed), None)
        record_crash("band2", exc, current_pmcid=current)
        raise
    finally:
        stop_live_watch()
        append_jsonl(RUNTIME_EVENTS_PATH, {
            "ts": utc_now(), "event": "band2_end_or_exception",
            "elapsed_s": round(time.time() - started, 3),
            "peak_rss_kb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
        })

    assert BAND2_MANIFEST.get("status") == "complete"
    events = read_jsonl(MODEL_CALL_EVENTS_PATH)
    successes = [e for e in events if e.get("result") == "success"]
    provider_errors = [e for e in events if e.get("result") != "success"]
    cost_summary = opus5_cost_summary(successes)
    model_summary = {
        "successful_provider_calls": len(successes),
        "provider_exceptions": len(provider_errors),
        "calls_by_stage": dict(collections.Counter(e.get("stage") for e in successes)),
        "input_tokens": sum(int(e.get("input_tokens") or 0) for e in successes),
        "output_tokens": sum(int(e.get("output_tokens") or 0) for e in successes),
        "cache_creation_input_tokens": sum(
            int(e.get("cache_creation_input_tokens") or 0) for e in successes),
        "cache_read_input_tokens": sum(
            int(e.get("cache_read_input_tokens") or 0) for e in successes),
        "prompt_cache_requested_calls": sum(
            e.get("prompt_cache_requested") is True for e in successes),
        "prompt_cache_hit_calls": sum(
            int(e.get("cache_read_input_tokens") or 0) > 0 for e in successes),
        "cost_summary": cost_summary,
        "outer_seam_receipt": receipt.summary(),
        "note": (
            "Provider-call totals come from the raw transport wrapper. The outer "
            "seam receipt is not used as a provider-call denominator because one "
            "coverage seam invocation may issue one request per atomic claim."
        ),
    }
    atomic_json(RUN_ROOT / "model_call_summary.json", model_summary)
    update_attempt_state("band2_complete", model_call_summary=model_summary)
    print("Band 2 model-call summary:", json.dumps(model_summary, indent=2))
    print(f"BAND 2: PASS in {(time.time()-started)/60:.1f} minutes")

assert BAND2_MANIFEST.get("evidence_scope_effective") == "fulltext_sections"
assert BAND2_MANIFEST.get("status") == "complete"

## Cell 9 — truthful current-chunk failure census

In [ ]:
PREDICTIONS_PATH = Path(BAND2_MANIFEST["predictions_path"])
rows = read_jsonl(PREDICTIONS_PATH)
row_ids = [row.get("citation_id") for row in rows]
assert len(row_ids) == len(set(row_ids))
assert set(row_ids) == EXPECTED_CITATION_IDS, {
    "missing": sorted(EXPECTED_CITATION_IDS - set(row_ids))[:10],
    "extra": sorted(set(row_ids) - EXPECTED_CITATION_IDS)[:10],
}

# Derive the live vocabulary from the pinned engine. It currently contains 14
# dispositions; the census refuses a row outside this vocabulary.
DISPOSITION_VOCAB = [
    judgment_run.DISP_EXCLUDED_NO_CITANCE,
    judgment_run.DISP_EXCLUDED_NO_CITED_PMID,
    judgment_run.DISP_EXCLUDED_PREBAND_MISSING,
    judgment_run.DISP_EXCLUDED_PREBAND,
    judgment_run.DISP_QUARANTINE_PARSE,
    judgment_run.DISP_HELD_NO_CLAIMS,
    judgment_run.DISP_PREDICTED,
    judgment_run.DISP_HELD_FULL_COVERAGE,
    judgment_run.DISP_HELD_INSUFFICIENT,
    judgment_run.DISP_HELD_PENDING_F5_F7,
    judgment_run.DISP_HELD_PROVENANCE_UNJUDGEABLE,
    judgment_run.DISP_HELD_STRENGTH_UNJUDGEABLE,
    judgment_run.DISP_HELD_COCITATION_COVERED,
    judgment_run.DISP_HELD_UNSUPPORTED_COCITATION_MEMBER,
]
assert len(DISPOSITION_VOCAB) == len(set(DISPOSITION_VOCAB))
observed_dispositions = collections.Counter(row.get("disposition") for row in rows)
unknown_dispositions = sorted(set(observed_dispositions) - set(DISPOSITION_VOCAB))
assert not unknown_dispositions, unknown_dispositions
disposition_counts = {name: observed_dispositions.get(name, 0)
                      for name in DISPOSITION_VOCAB}

EXCLUDED = {
    judgment_run.DISP_EXCLUDED_NO_CITANCE,
    judgment_run.DISP_EXCLUDED_NO_CITED_PMID,
    judgment_run.DISP_EXCLUDED_PREBAND_MISSING,
    judgment_run.DISP_EXCLUDED_PREBAND,
}
category_counts = {
    "excluded_before_judgment": sum(disposition_counts[x] for x in EXCLUDED),
    "quarantined_no_terminal_verdict":
        disposition_counts[judgment_run.DISP_QUARANTINE_PARSE],
    "held": sum(n for name, n in disposition_counts.items()
                if name.startswith("held_")),
    "terminal_prediction": disposition_counts[judgment_run.DISP_PREDICTED],
}
assert sum(category_counts.values()) == len(rows)
entered_band2 = len(rows) - category_counts["excluded_before_judgment"]

def quarantine_mode(error):
    text = error or ""
    if "Unterminated string" in text:
        return "truncated_json"
    if "Extra data" in text:
        return "multiple_json_objects"
    if "Expecting ',' delimiter" in text:
        return "malformed_delimiter"
    if "Expecting value" in text or "empty model output" in text:
        return "empty_or_non_json"
    if "JSON keys mismatch" in text:
        return "schema_keys"
    if "must be" in text or "duplicate" in text:
        return "schema_value"
    return "other_interpretation_failure"

quarantines = [
    row for row in rows
    if row.get("disposition") == judgment_run.DISP_QUARANTINE_PARSE]
quarantine_modes = collections.Counter(
    quarantine_mode(row.get("parse_error")) for row in quarantines)

model_events = read_jsonl(MODEL_CALL_EVENTS_PATH)
provider_success = [e for e in model_events if e.get("result") == "success"]
provider_error = [e for e in model_events if e.get("result") != "success"]
band2_provider_success = [
    e for e in provider_success if e.get("stage") != "band1_llm"]
response_parse_failure_rate = (
    len(quarantines) / len(band2_provider_success)
    if band2_provider_success else None)
reference_quarantine_rate = (
    len(quarantines) / entered_band2 if entered_band2 else None)

abstract_events = read_jsonl(ABSTRACT_EVENTS_PATH)
fulltext_events = read_jsonl(FULLTEXT_EVENTS_PATH)
transport_events = read_jsonl(TRANSPORT_EVENTS_PATH)
abstract_outcomes = collections.Counter(e.get("result") for e in abstract_events)
fulltext_outcomes = collections.Counter(e.get("result") for e in fulltext_events)
fulltext_incomplete_reasons = collections.Counter(
    reason for event in fulltext_events
    for reason in (event.get("incomplete_reasons") or []))
raw_transport_outcomes = collections.Counter(
    (e.get("stage"), e.get("result"), str(e.get("status_code")))
    for e in transport_events)

coverage_assessed_claims = sum(
    1 for row in rows for verdict in (row.get("coverage_verdicts") or [])
    if verdict.get("established") is not None)
f3_reached = sum(row.get("provenance") is not None for row in rows)
f4_reached = int((BAND2_MANIFEST.get("f4") or {}).get("eligible_claims") or 0)

dm = json.loads(DISPOSITION_MANIFEST_PATH.read_text(encoding="utf-8"))
attest = dm.get("check_attestations") or {}
preband_counts = dm.get("label_counts") or {}
finding_counts = collections.Counter(
    label for row in rows for label in labels_of(row))

# Positives are incidental, but when they occur the by-product preserves the
# complete adjudication record rather than a thin sentence-and-label digest.
positive_records = []
for row in rows:
    labels = labels_of(row)
    if labels:
        positive_records.append({
            "record_kind": "band2_full_record", "labels": labels,
            "record": row,
        })
for row in band1_logs:
    label = row.get("label")
    if label in {"F1", "F2"}:
        positive_records.append({
            "record_kind": "band1_full_log", "labels": [label],
            "record": row,
        })
    elif label == "F8":
        positive_records.append({
            "record_kind": "f8_publication_type_precursor",
            "labels": [],
            "precursor": "Retracted Publication",
            "taxonomy_complete_f8": False,
            "record": row,
        })
POSITIVES_PATH = RUN_ROOT / "positives_full_records.jsonl"
atomic_jsonl(POSITIVES_PATH, positive_records)

reachability = {
    "F1": {
        "configured": True, "reachable": True,
        "reached": int((attest.get("F1") or {}).get("answered") or 0),
        "unit": "references answered", "findings": int(preband_counts.get("F1", 0)),
    },
    "F2": {
        "configured": True, "reachable": True,
        "reached": int((attest.get("F2") or {}).get("answered") or 0),
        "unit": "references answered", "findings": int(preband_counts.get("F2", 0)),
    },
    "F3": {
        "configured": True, "reachable": True, "reached": f3_reached,
        "unit": "reference records reaching provenance",
        "findings": int(finding_counts.get("F3", 0)),
    },
    "F4": {
        "configured": True, "reachable": True, "reached": f4_reached,
        "unit": "claims formally assessed",
        "findings": int(finding_counts.get("F4", 0)),
    },
    "F5": {
        "configured": False, "reachable": False, "reached": 0,
        "unit": "references", "findings": 0,
        "reason": "no production evidence builder",
    },
    "F6": {
        "configured": True, "reachable": True,
        "reached": coverage_assessed_claims,
        "unit": "claims assessed against complete full text",
        "findings": int(finding_counts.get("F6", 0)),
    },
    "F7": {
        "configured": False, "reachable": False, "reached": 0,
        "unit": "references", "findings": 0,
        "reason": "no production evidence builder",
    },
    "F8": {
        "configured": False, "reachable": False, "reached": 0,
        "unit": "taxonomy-complete assessments", "findings": 0,
        "reason": "publication-type precursor exists; timing detector is absent",
        "precursor_answered": int((attest.get("F8") or {}).get("answered") or 0),
        "precursor_candidates": int(preband_counts.get("F8", 0)),
    },
}

per_paper = {}
for pmcid in CHUNK_PMCIDS:
    paper_rows = [row for row in rows if row.get("citing_pmcid") == pmcid]
    paper_entered = sum(row.get("disposition") not in EXCLUDED for row in paper_rows)
    paper_quarantine = sum(
        row.get("disposition") == judgment_run.DISP_QUARANTINE_PARSE
        for row in paper_rows)
    per_paper[pmcid] = {
        "input_profile": INPUT_PROFILES.get(pmcid),
        "records": len(paper_rows), "entered_band2": paper_entered,
        "quarantined": paper_quarantine,
        "quarantine_rate_among_entered":
            (paper_quarantine / paper_entered if paper_entered else None),
    }

integrity_failures = []
if not BAND2_MANIFEST.get("accounting_ok"):
    integrity_failures.append("manifest accounting_ok is false")
if int(BAND2_MANIFEST.get("chain_record_count") or 0) != len(rows):
    integrity_failures.append("hash-chain record count differs from predictions")
if not BAND2_MANIFEST.get("module_sha256_stable"):
    integrity_failures.append("governing module bytes changed during execution")
if not (BAND2_MANIFEST.get("executed_domain") or {}).get("matches_preflight"):
    integrity_failures.append("execution domain differs from preflight")
queue_audit = BAND2_MANIFEST.get("queue_audit") or {}
if not queue_audit.get("matches"):
    integrity_failures.append("annotation queue does not match scoreable records")

chunk_census = {
    "schema": "cre_soak_failure_census_chunk_v2",
    "created_at": utc_now(), "not_reportable": True,
    "diagnostic_rates_only": True,
    "strategy_version": STRATEGY_VERSION,
    "pmcid_list_sha256": drawn["pmcid_list_sha256"],
    "chunk_index": CHUNK_INDEX, "attempt_index": ATTEMPT_INDEX,
    "run_root": str(RUN_ROOT), "code_commit": CODE_COMMIT,
    "papers_requested": CHUNK_PMCIDS,
    "papers_parser_success": sorted(REFERENCE_COUNTS),
    "input_profiles": INPUT_PROFILES,
    "reference_records": len(rows),
    "disposition_vocabulary_size": len(DISPOSITION_VOCAB),
    "dispositions": disposition_counts,
    "categories": category_counts,
    "entered_band2": entered_band2,
    "engine_scoreable_records": BAND2_MANIFEST.get("scoreable_records"),
    "quarantine": {
        "count": len(quarantines), "modes": dict(quarantine_modes),
        "rate_among_entered_references": reference_quarantine_rate,
        "rate_per_successful_band2_provider_response": response_parse_failure_rate,
        "rate_note": (
            "The first rate is a reference-terminal rate. The second uses the raw "
            "provider-call log; it is not divided by outer seam invocations."
        ),
    },
    "provider": {
        "successful_calls": len(provider_success),
        "exceptions": len(provider_error),
        "calls_by_stage": dict(collections.Counter(
            e.get("stage") for e in provider_success)),
        "input_tokens": sum(int(e.get("input_tokens") or 0) for e in provider_success),
        "output_tokens": sum(int(e.get("output_tokens") or 0) for e in provider_success),
    },
    "transport": {
        "abstract_outcomes": dict(abstract_outcomes),
        "fulltext_outcomes": dict(fulltext_outcomes),
        "fulltext_incomplete_reasons": dict(fulltext_incomplete_reasons),
        "raw_http_outcomes": {
            "|".join(key): value for key, value in raw_transport_outcomes.items()
        },
    },
    "reachability": reachability,
    "finding_counts": dict(finding_counts),
    "positives_byproduct": {
        "count": len(positive_records), "path": str(POSITIVES_PATH),
        "sha256": sha256_file(POSITIVES_PATH),
        "note": (
            "Complete source records are preserved. F8 publication-type rows are "
            "named precursors and never emitted as taxonomy-complete F8 findings."
        ),
    },
    "sentence_partition_diagnostics":
        BAND2_MANIFEST.get("sentence_partition_diagnostics") or {},
    "per_paper": per_paper,
    "integrity_failures": integrity_failures,
    "prediction_ids_sha256": sha256_bytes("\n".join(sorted(row_ids)).encode()),
    "predictions_sha256": sha256_file(PREDICTIONS_PATH),
    "manifest_sha256": sha256_file(BAND2_MANIFEST_PATH),
}

CHUNK_CENSUS_PATH = RUN_ROOT / "chunk_census.json"
atomic_json(CHUNK_CENSUS_PATH, chunk_census)

print("REACHABILITY COMES BEFORE COUNTS")
print("stratum | configured | reachable | reached | unit | findings | interpretation")
for label in [f"F{i}" for i in range(1, 9)]:
    row = reachability[label]
    if not row["reachable"]:
        interpretation = "NOT RUN — " + row.get("reason", "unwired")
    elif row["reached"] == 0:
        interpretation = "WIRED BUT NOT REACHED — zero is not a pass"
    elif row["findings"] == 0:
        interpretation = f"performed on {row['reached']} {row['unit']}; found none"
    else:
        interpretation = f"{row['findings']} finding(s) after reaching {row['reached']}"
    print(" | ".join(map(str, [
        label, row["configured"], row["reachable"], row["reached"],
        row["unit"], row["findings"], interpretation,
    ])))

print(f"\nDISPOSITION CENSUS — {len(DISPOSITION_VOCAB)} live dispositions")
for name in DISPOSITION_VOCAB:
    print(f"  {name:44s} {disposition_counts[name]:6d}")
print("categories:", json.dumps(category_counts, sort_keys=True))
print("entered Band 2:", entered_band2)
print("quarantine modes:", json.dumps(dict(quarantine_modes), sort_keys=True))
print("provider calls/errors:", len(provider_success), len(provider_error))
print("abstract outcomes:", json.dumps(dict(abstract_outcomes), sort_keys=True))
print("fulltext outcomes:", json.dumps(dict(fulltext_outcomes), sort_keys=True))
print("fulltext incomplete reasons:",
      json.dumps(dict(fulltext_incomplete_reasons), sort_keys=True))
print("integrity failures:", integrity_failures)
print("full-record positives by-product:", POSITIVES_PATH, len(positive_records))
print("chunk census:", CHUNK_CENSUS_PATH)

if integrity_failures:
    update_attempt_state(
        "failed", failed_stage="chunk_census_integrity",
        integrity_failures=integrity_failures)
    raise RuntimeError("INTEGRITY FAILURE — do not continue to another chunk")
update_attempt_state("complete", chunk_census=str(CHUNK_CENSUS_PATH))
if reference_quarantine_rate is not None and reference_quarantine_rate > 0.20:
    raise RuntimeError(
        f"STOP: {reference_quarantine_rate:.1%} of entered references quarantined")
complete_fulltext = fulltext_outcomes.get("complete", 0)
attempted_fulltext = complete_fulltext + fulltext_outcomes.get("incomplete", 0)
if attempted_fulltext and complete_fulltext / attempted_fulltext < 0.50:
    raise RuntimeError("STOP: fewer than half of attempted cited full texts were complete")

## Cell 10 — cross-chunk failure census and endurance report

In [ ]:
# Select exactly one completed attempt per chunk. Failed attempts remain in the crash
# and provider-cost census but never duplicate reference dispositions.
complete_by_chunk = {}
duplicate_complete_attempts = {}
for census_path in sorted(DRAW_ROOT.glob("chunk_*/attempt_*/chunk_census.json")):
    census = json.loads(census_path.read_text(encoding="utf-8"))
    state_path = census_path.parent / "attempt_state.json"
    state = (json.loads(state_path.read_text(encoding="utf-8"))
             if state_path.exists() else {})
    if census.get("integrity_failures") or state.get("status") != "complete":
        continue
    index = int(census["chunk_index"])
    if index in complete_by_chunk:
        duplicate_complete_attempts.setdefault(index, []).append(str(census_path))
        continue
    complete_by_chunk[index] = (census_path, census)

selected = [complete_by_chunk[i] for i in sorted(complete_by_chunk)]
aggregate_dispositions = collections.Counter({name: 0 for name in DISPOSITION_VOCAB})
aggregate_categories = collections.Counter()
aggregate_findings = collections.Counter()
aggregate_quarantine_modes = collections.Counter()
aggregate_abstract = collections.Counter()
aggregate_fulltext = collections.Counter()
aggregate_fulltext_reasons = collections.Counter()
aggregate_reach = {
    f"F{i}": {"reached": 0, "findings": 0} for i in range(1, 9)
}
all_prediction_ids = set()
duplicate_prediction_ids = []
selected_attempt_roots = []
aggregate_per_paper = {}
duplicate_papers = []

for census_path, census in selected:
    selected_attempt_roots.append(census_path.parent)
    aggregate_dispositions.update(census["dispositions"])
    aggregate_categories.update(census["categories"])
    aggregate_findings.update(census["finding_counts"])
    aggregate_quarantine_modes.update(census["quarantine"]["modes"])
    aggregate_abstract.update(census["transport"]["abstract_outcomes"])
    aggregate_fulltext.update(census["transport"]["fulltext_outcomes"])
    aggregate_fulltext_reasons.update(
        census["transport"]["fulltext_incomplete_reasons"])
    for label, row in census["reachability"].items():
        aggregate_reach[label]["reached"] += int(row.get("reached") or 0)
        aggregate_reach[label]["findings"] += int(row.get("findings") or 0)
    for pmcid, paper in census.get("per_paper", {}).items():
        if pmcid in aggregate_per_paper:
            duplicate_papers.append(pmcid)
        aggregate_per_paper[pmcid] = paper
    manifest = json.loads(
        (census_path.parent / "band2" / "judgment_run_manifest.json")
        .read_text(encoding="utf-8"))
    for record in read_jsonl(manifest["predictions_path"]):
        citation_id = record.get("citation_id")
        if citation_id in all_prediction_ids:
            duplicate_prediction_ids.append(citation_id)
        all_prediction_ids.add(citation_id)

all_crashes = []
for events_path in DRAW_ROOT.glob("chunk_*/attempt_*/crash_events.jsonl"):
    all_crashes.extend(read_jsonl(events_path))
all_attempt_model_events = []
for events_path in DRAW_ROOT.glob("chunk_*/attempt_*/model_call_events.jsonl"):
    all_attempt_model_events.extend(read_jsonl(events_path))
all_attempt_transport_events = []
for events_path in DRAW_ROOT.glob("chunk_*/attempt_*/transport_events.jsonl"):
    all_attempt_transport_events.extend(read_jsonl(events_path))
all_attempt_abstract_events = []
for events_path in DRAW_ROOT.glob("chunk_*/attempt_*/abstract_evidence_events.jsonl"):
    all_attempt_abstract_events.extend(read_jsonl(events_path))
all_attempt_fulltext_events = []
for events_path in DRAW_ROOT.glob("chunk_*/attempt_*/fulltext_evidence_events.jsonl"):
    all_attempt_fulltext_events.extend(read_jsonl(events_path))
selected_model_events = []
for attempt_root in selected_attempt_roots:
    selected_model_events.extend(read_jsonl(attempt_root / "model_call_events.jsonl"))

selected_success = [e for e in selected_model_events if e.get("result") == "success"]
selected_band2_success = [
    e for e in selected_success if e.get("stage") != "band1_llm"]
all_success = [e for e in all_attempt_model_events if e.get("result") == "success"]
all_provider_errors = [
    e for e in all_attempt_model_events if e.get("result") != "success"]

input_profiles = []
integrity_failures = []
for _, census in selected:
    input_profiles.extend(census["input_profiles"].values())
    integrity_failures.extend(census.get("integrity_failures") or [])
if duplicate_prediction_ids:
    integrity_failures.append(
        f"duplicate citation ids across selected chunks: {duplicate_prediction_ids[:10]}")
if duplicate_papers:
    integrity_failures.append(
        f"duplicate papers across selected chunks: {sorted(set(duplicate_papers))[:10]}")

completed_chunks = len(selected)
status = "complete" if completed_chunks == n_chunks else "partial"
reference_counts_observed = sorted(
    int(row.get("reference_count") or 0) for row in input_profiles)
zero_reference_papers = [
    row["pmcid"] for row in input_profiles
    if row.get("input_failure") == "zero_references"]
input_failed_papers = [
    row["pmcid"] for row in input_profiles
    if row.get("parse_status") != "parsed"]

aggregate = {
    "schema": "cre_soak_failure_census_overall_v2",
    "created_at": utc_now(), "status": status,
    "not_reportable": True, "diagnostic_rates_only": True,
    "strategy_version": STRATEGY_VERSION,
    "corpus_seed": CORPUS_SEED,
    "pmcid_list_sha256": drawn["pmcid_list_sha256"],
    "expected_papers": TOTAL_PAPERS, "expected_chunks": n_chunks,
    "completed_chunks": completed_chunks,
    "missing_chunks": sorted(set(range(n_chunks)) - set(complete_by_chunk)),
    "duplicate_complete_attempts_ignored": duplicate_complete_attempts,
    "unique_reference_records": len(all_prediction_ids),
    "disposition_vocabulary_size": len(DISPOSITION_VOCAB),
    "dispositions": dict(aggregate_dispositions),
    "categories": dict(aggregate_categories),
    "finding_counts": dict(aggregate_findings),
    "quarantine": {
        "count": int(aggregate_categories.get(
            "quarantined_no_terminal_verdict", 0)),
        "modes": dict(aggregate_quarantine_modes),
        "entered_band2": sum(
            int(census.get("entered_band2") or 0) for _, census in selected),
        "rate_among_entered_references": (
            int(aggregate_categories.get("quarantined_no_terminal_verdict", 0))
            / sum(int(census.get("entered_band2") or 0)
                  for _, census in selected)
            if sum(int(census.get("entered_band2") or 0)
                   for _, census in selected) else None
        ),
        "rate_per_successful_band2_provider_response": (
            int(aggregate_categories.get("quarantined_no_terminal_verdict", 0))
            / len(selected_band2_success)
            if selected_band2_success else None
        ),
    },
    "per_paper": aggregate_per_paper,
    "transport": {
        "abstract_outcomes": dict(aggregate_abstract),
        "fulltext_outcomes": dict(aggregate_fulltext),
        "fulltext_incomplete_reasons": dict(aggregate_fulltext_reasons),
        "all_attempt_raw_http_outcomes": {
            "|".join(key): value for key, value in collections.Counter(
                (e.get("stage"), e.get("result"), str(e.get("status_code")))
                for e in all_attempt_transport_events).items()
        },
        "all_attempt_abstract_outcomes": dict(collections.Counter(
            e.get("result") for e in all_attempt_abstract_events)),
        "all_attempt_fulltext_outcomes": dict(collections.Counter(
            e.get("result") for e in all_attempt_fulltext_events)),
    },
    "reachability": aggregate_reach,
    "input_shape": {
        "papers_profiled": len(input_profiles),
        "input_failed_papers": input_failed_papers,
        "zero_reference_papers": zero_reference_papers,
        "citation_styles": dict(collections.Counter(
            row.get("citation_style", "unknown") for row in input_profiles)),
        "selection_strata": dict(collections.Counter(
            row.get("selection_stratum", "unknown") for row in input_profiles)),
        "references": sum(reference_counts_observed),
        "reference_count_distribution": {
            "min": min(reference_counts_observed) if reference_counts_observed else None,
            "median": (statistics.median(reference_counts_observed)
                       if reference_counts_observed else None),
            "max": max(reference_counts_observed) if reference_counts_observed else None,
        },
        "missing_pmid": sum(int(row.get("missing_pmid") or 0)
                            for row in input_profiles),
        "missing_citance": sum(int(row.get("missing_citance") or 0)
                               for row in input_profiles),
        "corporate_first_author": sum(
            int(row.get("corporate_first_author") or 0) for row in input_profiles),
        "non_ascii_titles": sum(int(row.get("non_ascii_titles") or 0)
                                for row in input_profiles),
        "sentence_partition_events": sum(
            int(row.get("sentence_partition_events") or 0)
            for row in input_profiles),
        "sentence_partition_uncovered_characters": sum(
            int(row.get("sentence_partition_uncovered_characters") or 0)
            for row in input_profiles),
    },
    "provider": {
        "selected_successful_calls": len(selected_success),
        "all_attempt_successful_calls_including_retries_after_crash": len(all_success),
        "all_attempt_provider_exceptions": len(all_provider_errors),
        "selected_input_tokens": sum(
            int(e.get("input_tokens") or 0) for e in selected_success),
        "selected_output_tokens": sum(
            int(e.get("output_tokens") or 0) for e in selected_success),
        "all_attempt_input_tokens": sum(
            int(e.get("input_tokens") or 0) for e in all_success),
        "all_attempt_output_tokens": sum(
            int(e.get("output_tokens") or 0) for e in all_success),
    },
    "endurance": {
        "runtime_events": sum(
            len(read_jsonl(root / "runtime_events.jsonl"))
            for root in selected_attempt_roots),
        "total_band2_elapsed_s": sum(
            float(event.get("elapsed_s") or 0)
            for root in selected_attempt_roots
            for event in read_jsonl(root / "runtime_events.jsonl")
            if event.get("event") == "band2_end_or_exception"),
        "max_peak_rss_kb": max(
            [int(event.get("peak_rss_kb") or 0)
             for root in selected_attempt_roots
             for event in read_jsonl(root / "runtime_events.jsonl")] or [0]),
        "in_progress_manifests_preserved": [
            str(path) for path in DRAW_ROOT.glob(
                "chunk_*/attempt_*/band2/judgment_run_manifest.json")
            if json.loads(path.read_text(encoding="utf-8")).get("status")
               == "in_progress"
        ],
    },
    "crashes": {
        "count": len(all_crashes),
        "by_stage": dict(collections.Counter(
            event.get("stage") for event in all_crashes)),
        "events_paths": [
            str(path) for path in DRAW_ROOT.glob(
                "chunk_*/attempt_*/crash_events.jsonl")
        ],
    },
    "integrity_failures": integrity_failures,
    "citation_id_set_sha256":
        sha256_bytes("\n".join(sorted(all_prediction_ids)).encode()),
    "note": (
        "All fractions are diagnostics of this deliberately adversarial soak corpus. "
        "They are not population rates, prevalence, or precision."
    ),
}

OVERALL_CENSUS_PATH = DRAW_ROOT / "soak_failure_census.json"
atomic_json(OVERALL_CENSUS_PATH, aggregate)

print("OVERALL REACHABILITY COMES BEFORE FINDING COUNTS")
print("stratum | reachable in configured system? | reached | findings | interpretation")
for label in [f"F{i}" for i in range(1, 9)]:
    if label in {"F5", "F7", "F8"}:
        reachable = False
        explanation = {
            "F5": "NOT RUN — no production evidence builder",
            "F7": "NOT RUN — no production evidence builder",
            "F8": "NOT IMPLEMENTED — publication-type lookup is only a precursor",
        }[label]
    else:
        reachable = True
        reached = aggregate_reach[label]["reached"]
        explanation = (
            f"performed on {reached} units"
            if reached else "WIRED BUT NOT REACHED — zero is not a pass")
    print(" | ".join(map(str, [
        label, reachable, aggregate_reach[label]["reached"],
        aggregate_reach[label]["findings"], explanation,
    ])))

print("\nSOAK CENSUS STATUS:", status.upper())
print(f"completed chunks: {completed_chunks}/{n_chunks}")
print("missing chunks:", aggregate["missing_chunks"])
print("unique reference records:", len(all_prediction_ids))
print("dispositions:", json.dumps(dict(aggregate_dispositions), sort_keys=True))
print("categories:", json.dumps(dict(aggregate_categories), sort_keys=True))
print("quarantine census:", json.dumps(aggregate["quarantine"], sort_keys=True))
print("input-shape census:", json.dumps(aggregate["input_shape"], sort_keys=True))
print("transport census:", json.dumps(aggregate["transport"], sort_keys=True))
print("endurance:", json.dumps(aggregate["endurance"], sort_keys=True))
print("crashes:", json.dumps(aggregate["crashes"], sort_keys=True))
print("provider telemetry:", json.dumps(aggregate["provider"], sort_keys=True))
print("integrity failures:", integrity_failures)
print("durable overall census:", OVERALL_CENSUS_PATH)

if integrity_failures:
    raise RuntimeError("OVERALL INTEGRITY FAILURE — stop the soak test")
if status == "complete":
    print("SOAK TEST DELIVERABLE: COMPLETE")
else:
    print("SOAK TEST DELIVERABLE: PARTIAL — continue with the first missing CHUNK_INDEX")

## Continue safely

For the next chunk, set CHUNK_INDEX in Cell 3 to the first missing chunk printed by Cell 10, then rerun Cells 3–10.

Nothing in this notebook deletes or overwrites a failed paid attempt. Shared source caches contain only validated successful retrievals. A failed attempt remains available for the crash, provider, and transport census.

The final artifact is soak_failure_census.json under the hash-bound draw directory. Its status is COMPLETE only after every expected chunk has exactly one selected completed attempt and no duplicate citation IDs or integrity failures.